In [2]:
import os
import warnings
warnings.filterwarnings("ignore")
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent, ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph, MessagesState
from langchain_core.tools import tool
from typing import Literal
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [23]:
load_dotenv()

True

In [41]:
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(model='gpt-4o-mini')
parser = StrOutputParser()

Custom Tool

In [5]:
@tool
def get_word_length(word: str) -> int:
    """Returns the length of the word"""
    return len(word)

In [6]:
get_word_length.invoke("Shubham")

7

Predefined Tools

In [7]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
api_wrapper = WikipediaAPIWrapper()
tool = WikipediaQueryRun(api_wrapper=api_wrapper)
print(tool.run({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Retrieval-augmented generation
Summary: Retrieval-augmented generation (RAG) is a technique that enables generative artificial intelligence (Gen AI) models to retrieve and incorporate new information. It modifies interactions with a large language model (LLM) so that the model responds to user queries with reference to a specified set of documents, using this information to supplement information from its pre-existing training data. This allows LLMs to use domain-specific and/or updated information. Use cases include providing chatbot access to internal company data or generating responses based on authoritative sources.
RAG improves la

In [8]:
from langchain_community.tools.tavily_search import TavilySearchResults
search = TavilySearchResults()
search.invoke("what is the weather in SF")

[{'title': 'Monday, April 14, 2025. San Francisco, CA - Weather Forecast',
  'url': 'https://weathershogun.com/weather/usa/ca/san-francisco/480/april/2025-04-14',
  'content': 'San Francisco, California Weather: Monday, April 14, 2025. Cloudy weather, overcast skies with clouds. Day 66°. Night 54°. Precipitation 0 %.',
  'score': 0.94584644},
 {'title': 'Weather in San Francisco',
  'url': 'https://www.weatherapi.com/',
  'content': "{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1744669227, 'localtime': '2025-04-14 15:20'}, 'current': {'last_updated_epoch': 1744668900, 'last_updated': '2025-04-14 15:15', 'temp_c': 12.2, 'temp_f': 54.0, 'is_day': 1, 'condition': {'text': 'Mist', 'icon': '//cdn.weatherapi.com/weather/64x64/day/143.png', 'code': 1030}, 'wind_mph': 11.0, 'wind_kph': 17.6, 'wind_degree': 255, 'wind_dir': 'WSW', 'pressure_mb': 1016.0, 'p

Create Agents using predefined tool

In [9]:
from langchain import hub
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

In [10]:
tools = [search]
prompt=hub.pull("hwchase17/openai-functions-agent")
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [11]:
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [12]:
agent_executor.invoke({"input": "hello how are you?"})



> Entering new AgentExecutor chain...
Hello! I'm here and ready to assist you. How can I help you today?

> Finished chain.


{'input': 'hello how are you?',
 'output': "Hello! I'm here and ready to assist you. How can I help you today?"}

In [13]:
agent_executor.invoke({"input": "What is the current weather in Raleigh, NC?"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'current weather in Raleigh, NC'}`


[{'title': 'Monday, April 14, 2025. Raleigh, NC - Weather Forecast', 'url': 'https://weathershogun.com/weather/usa/nc/raleigh/3375/april/2025-04-14', 'content': 'Raleigh, North Carolina Weather: Monday, April 14, 2025. Cloudy weather, overcast skies with clouds. Day 73°. Night 57°. Precipitation 7 %.', 'score': 0.9489978}, {'title': 'Weather in Raleigh, NC', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Raleigh', 'region': 'North Carolina', 'country': 'United States of America', 'lat': 35.7719, 'lon': -78.6389, 'tz_id': 'America/New_York', 'localtime_epoch': 1744669621, 'localtime': '2025-04-14 18:27'}, 'current': {'last_updated_epoch': 1744668900, 'last_updated': '2025-04-14 18:15', 'temp_c': 28.9, 'temp_f': 84.0, 'is_day': 1, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/day/116.png', 'code

{'input': 'What is the current weather in Raleigh, NC?',
 'output': 'The current weather in Raleigh, NC is as follows:\n- Temperature: 84.0°F (28.9°C)\n- Condition: Partly cloudy\n- Humidity: 36%\n- Wind: 18.4 km/h from WSW\n- Cloud cover: 50%\n\nIf you need more detailed information or updates, please check the [Weather API website](https://www.weatherapi.com/).'}

RAG Agent

In [14]:
from langchain.tools.retriever import create_retriever_tool

In [15]:
loader = WebBaseLoader("https://docs.smith.langchain.com/overview")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
).split_documents(docs)
vector = FAISS.from_documents(documents,embeddings)
retriever = vector.as_retriever()

In [16]:
retriever_tool = create_retriever_tool(
    retriever,
    "langgraph_search",
    "Search for information about LangGraph. For any questions about LangGraph, you must use this tool!",
)

In [17]:
tools = [search, retriever_tool]
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [18]:
agent_executor.invoke({"input": "hi! what is a LangGraph?"})



> Entering new AgentExecutor chain...

Invoking: `langgraph_search` with `{'query': 'LangGraph'}`


🦜️🛠️ LangSmith






Skip to main contentJoin us at  Interrupt: The Agent AI Conference by LangChain on May 13 & 14 in San Francisco!API ReferenceRESTPythonJS/TSSearchRegionUSEUGo to AppPage Not FoundWe could not find what you were looking for.Head back to our main docs page or use the search bar to find the page you need.CommunityDiscordTwitterGitHubDocs CodeLangSmith SDKPythonJS/TSMoreHomepageBlogLangChain Python DocsLangChain JS/TS DocsCopyright © 2025 LangChain, Inc.LangGraph is not directly mentioned in the search results. Would you like me to look for more information related to LangGraph?

> Finished chain.


{'input': 'hi! what is a LangGraph?',
 'output': 'LangGraph is not directly mentioned in the search results. Would you like me to look for more information related to LangGraph?'}

In [19]:
agent_executor.invoke({"input": "Hi! What is the weather today in Celcius?"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'weather today in Celsius'}`


[{'title': 'United States weather forecast for Monday, 14 April 2025', 'url': 'https://www.weather25.com/north-america/usa?page=date&date=14-4', 'content': '| Sun | Mon | Tue | Wed | Thu | Fri | Sat |\n| --- | --- | --- | --- | --- | --- | --- |\n| \n | \n| 1  53° /39° | 2  48° /33° | 3  50° /35° | 4  51° /37° | 5  55° /39° |\n| 6  59° /41° | 7  57° /41° | 8  59° /42° | 9  57° /41° | 10  55° /39° | 11  59° /41° | 12  60° /42° |\n| 13  60° /46° | 14  69° /46° | 15  59° /46° | 16  57° /42° | 17  53° /37° | 18  50° /39° | 19  57° /37° |\n| 20  60° /42° | 21  60° /39° | 22  57° /37° | 23  57° /37° | 24  59° /41° | 25  59° /39° | 26  59° /39° | [...] | Light rain shower | 12 Miles |\n| 2021 - April 14 | 60 ° / 44 ° | 0.02 in | \n\n| Light rain shower | 8 Miles |\n| 2020 - April 14 | 51 ° / 39 ° | 0 in | \n\n| Patchy rain possible | 12 Miles |\nWeather Calendar\n\nAp

{'input': 'Hi! What is the weather today in Celcius?',
 'output': "I couldn't find the specific weather in Celsius for today. You may want to check a reliable weather website or app for the most up-to-date information."}

Create ReAct Agent

In [20]:
from langchain.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from langchain_core.prompts import PromptTemplate
from langchain.agents import AgentExecutor, create_react_agent

In [24]:
google_search = GoogleSerperAPIWrapper()
tools = [
    Tool(
        name="Intermediate Answer",
        func=google_search.run,
        description="useful for when you need to ask with search",
        verbose=True
    )
]

In [25]:
template = '''Answer the following questions as best you can. You have access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought:{agent_scratchpad}'''

In [26]:
prompt = PromptTemplate.from_template(template)
search_agent = create_react_agent(llm,tools,prompt)
agent_executor = AgentExecutor(
    agent=search_agent,
    tools=tools,
    verbose=True,
    return_intermediate_steps=True,
)

In [27]:
agent_executor.invoke({"input": "Who has scored the most number of ODI centuries in cricket and the numbe of matches he has played?"})



> Entering new AgentExecutor chain...
Let's use the tools provided to find the answer to this question.
Action: Intermediate Answer
Action Input: "cricketer with most ODI centuries and number of matches played"Most hundreds in a career in ODIs ; MJ Guptill (NZ). 2009-2022, 198 ; ME Waugh (AUS). 1988-2002, 244 ; SD Hope (WI). 2016-2024, 133 ; AJ Finch (AUS). 2013-2022, 146 ... To date, 127 cricketers have scored 15 or more international centuries, 83 of whom went on to score 20 or more centuries, 44 have scored 30 or more centuries, ... As of March 2025, Virat Kohli ranked first for the most centuries scored in men's One Day International cricket, with 51 ODI tons to his name. ... Virat Kohli and Sachin Tendulkar have the most ODI centuries in history with 49 each. How many centuries do Virat Kohli and Rohit Sharma have? +. Most hundreds against one team in ODIs ; V Kohli (IND). 2008-2024, 56 ; V Kohli (IND). 2009-2023, 43 ; SR Tendulkar (IND). 1991-2012, 71 ; RG Sharma (IND). 2007- .

{'input': 'Who has scored the most number of ODI centuries in cricket and the numbe of matches he has played?',
 'output': 'Virat Kohli has scored the most number of ODI centuries in cricket with 50 centuries.',
 'intermediate_steps': [(AgentAction(tool='Intermediate Answer', tool_input='cricketer with most ODI centuries and number of matches played', log='Let\'s use the tools provided to find the answer to this question.\nAction: Intermediate Answer\nAction Input: "cricketer with most ODI centuries and number of matches played"'),
   "Most hundreds in a career in ODIs ; MJ Guptill (NZ). 2009-2022, 198 ; ME Waugh (AUS). 1988-2002, 244 ; SD Hope (WI). 2016-2024, 133 ; AJ Finch (AUS). 2013-2022, 146 ... To date, 127 cricketers have scored 15 or more international centuries, 83 of whom went on to score 20 or more centuries, 44 have scored 30 or more centuries, ... As of March 2025, Virat Kohli ranked first for the most centuries scored in men's One Day International cricket, with 51 ODI t

ReAct with Custom Tool

In [33]:
from langchain.tools import tool
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent

In [34]:
@tool
def get_employee_id(name):
  """
  To get employee id, it takes employee name as arguments
  name(str): Name of the employee
  """
  fake_employees = {
    "Alice": "E001",
    "Bob": "E002",
    "Charlie": "E003",
    "Diana": "E004",
    "Evan": "E005",
    "Fiona": "E006",
    "George": "E007",
    "Hannah": "E008",
    "Ian": "E009",
    "Jasmine": "E010"}
  
  return fake_employees.get(name,"Employee not found")

In [35]:
@tool
def get_employee_salary(employee_id):
  """
  To get the salary of an employee, it takes employee_id as input and return salary
  """
  employee_salaries = {
    "E001": 56000,
    "E002": 47000,
    "E003": 52000,
    "E004": 61000,
    "E005": 45000,
    "E006": 58000,
    "E007": 49000,
    "E008": 53000,
    "E009": 50000,
    "E010": 55000
    }
  return employee_salaries.get(employee_id,"Employee not found")

In [ ]:
prompt = hub.pull("hwchase17/react")
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [42]:
tools = [get_employee_salary, get_employee_id]
agent = create_react_agent(llm,tools,prompt)
agent_executor = AgentExecutor(agent=agent,tools=tools,verbose=True)

In [43]:
agent_executor.invoke({"input":"What is the Salary of Evan?"})



> Entering new AgentExecutor chain...
To find the salary of Evan, I first need to get Evan's employee ID. 

Action: get_employee_id  
Action Input: "Evan"  E005Now that I have Evan's employee ID (E005), I can retrieve his salary.

Action: get_employee_salary  
Action Input: "E005"  45000I now know the final answer.  
Final Answer: The salary of Evan is 45000.

> Finished chain.


{'input': 'What is the Salary of Evan?',
 'output': 'The salary of Evan is 45000.'}

Self-Ask Agent

In [44]:
from langchain.agents import create_self_ask_with_search_agent

In [45]:
google_search = GoogleSerperAPIWrapper()
tools = [
    Tool(
        name="Intermediate Answer",
        func=google_search.run,
        description="useful for when you need to ask with search",
        verbose=True
    )
]

In [46]:
prompt = hub.pull("hwchase17/self-ask-with-search")
print(prompt.template)

Question: Who lived longer, Muhammad Ali or Alan Turing?
Are follow up questions needed here: Yes.
Follow up: How old was Muhammad Ali when he died?
Intermediate answer: Muhammad Ali was 74 years old when he died.
Follow up: How old was Alan Turing when he died?
Intermediate answer: Alan Turing was 41 years old when he died.
So the final answer is: Muhammad Ali

Question: When was the founder of craigslist born?
Are follow up questions needed here: Yes.
Follow up: Who was the founder of craigslist?
Intermediate answer: Craigslist was founded by Craig Newmark.
Follow up: When was Craig Newmark born?
Intermediate answer: Craig Newmark was born on December 6, 1952.
So the final answer is: December 6, 1952

Question: Who was the maternal grandfather of George Washington?
Are follow up questions needed here: Yes.
Follow up: Who was the mother of George Washington?
Intermediate answer: The mother of George Washington was Mary Ball Washington.
Follow up: Who was the father of Mary Ball Washin

In [47]:
agent = create_self_ask_with_search_agent(llm,tools,prompt)
agent_executor = AgentExecutor(agent=agent,tools=tools,verbose=True,handle_parsing_errors=True)

In [49]:
agent_executor.invoke({"input":"Who is the current Test captain of the indian cricket team?"})



> Entering new AgentExecutor chain...
Yes.  
Follow up: Who is the current captain of the Indian Test cricket team?  Rohit Sharma is current captain of the men's team in Tests and ODIs and Harmanpreet Kaur is current captain of women's team in all Formats. Soon after taking up the captaincy, Dhoni led the team to the inaugural T20 World Cup title. Anil Kumble was appointed Test captain in November 2007, but ... Captain: Rohit Sharma (Tests, ODIs), Suryakumar Yadav (T20Is) ; Coach: Gautam Gambhir (head coach), Morne Morkel (bowling coach), Sitanshu Kotak (batting coach), ... Jasprit Bumrah will lead the team in the fifth and final Test Test against Australia at the Sydney Cricket Ground. The Indian national team is currently captained by Rohit Sharma in Tests and ODIs. India will return to Test action with a series against Bangladesh in ... List of Captains for India in Tests ; JJ Bumrah. 2022-2025, 3 ; R Dravid. 2003-2007, 25 ; RG Sharma. 2022-2024, 24 ; MS Dhoni. 2008-2014, 60 ... .

{'input': 'Who is the current Test captain of the indian cricket team?',
 'output': 'Rohit Sharma'}